In [1]:
PALETTE = [
    "#5F4690",
    "#1D6996",
    "#38A6A5",
    "#0F8554",
    "#73AF48",
    "#EDAD08",
    "#E17C05",
    "#CC503E",
    "#94346E",
    "#6F4070",
    "#994E95",
    "#FF89F9",
    "#FBC5F9",
    "#666666",
]

RESULTS_PATH = "eval"
STATS_PATH = "../../dataset/geminin_drugs_stats.json"
MARKER_SIZE = 3

In [3]:
from pathlib import Path
import pandas as pd

def load_results(res_dir):
    eval_dir = Path(res_dir)

    dfs = []

    for model_dir in eval_dir.iterdir():
        if model_dir.is_dir():
            csv_path = model_dir / "downstream_results.csv"
            if csv_path.exists():
                df = pd.read_csv(csv_path)
                df["model"] = model_dir.name  # optional: keep model name
                dfs.append(df)

    all_results = pd.concat(dfs, ignore_index=True)

    return all_results

In [4]:
df = load_results(RESULTS_PATH)

print(df.head())

      target feature  accuracy      loss  attention_entropy  num_frames  \
0  cell_type     all     0.722  0.621808           6.492233           1   
1  cell_type    rand     0.549  1.016030           0.000000           1   
2  cell_type     1ch     0.505  1.146223           5.506904           1   
3  treatment     all     0.288  2.074268           6.551058           1   
4  treatment    rand     0.266  2.227040           0.000000           1   

                     model  
0  rvm_500K_multi_Ch_smr05  
1  rvm_500K_multi_Ch_smr05  
2  rvm_500K_multi_Ch_smr05  
3  rvm_500K_multi_Ch_smr05  
4  rvm_500K_multi_Ch_smr05  


In [9]:
df_4frames = df[df["num_frames"] == 4]
table = df_4frames.pivot_table(
    values="accuracy",
    index=["target", "feature"],
    columns=["model"],
    aggfunc="mean",  # use "first" if each combination is unique
)

table.style.format("{:.3f}")

In [7]:
import json

with open(STATS_PATH) as f:
    data = json.load(f)

# Find all experiment/channel pairs with n == 0
to_null = []

for key, value in data.items():
    if key.endswith("/n") and value == 0:
        prefix = key[:-2]  # remove "/n"
        to_null.append(prefix)

# Set all statistics for those pairs to None
for prefix in to_null:
    for stat in ("mean", "std", "n"):
        key = f"{prefix}/{stat}"
        if key in data:
            data[key] = None

# Save the modified JSON
with open(STATS_PATH, "w") as f:
    json.dump(data, f, indent=2)

In [10]:
import json

with open(STATS_PATH) as f:
    data = json.load(f)

import pandas as pd

records = []

for key, value in data.items():
    experiment, channel, stat = key.split("/")
    records.append((experiment, channel, stat, value))

df_stats = pd.DataFrame(
    records,
    columns=["experiment", "channel", "stat", "value"]
)

df_stats_no_n = df_stats[df_stats["stat"] != "n"]

table = df_stats_no_n.pivot_table(
    values="value",
    index=["experiment"],
    columns=["channel", "stat"],
    aggfunc="first",
)

table.style.format("{:.3f}")